In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Data Reading**

In [0]:
df=spark.read.format('delta')\
    .load('abfss://bronze@adventwork.dfs.core.windows.net/returns')

In [0]:
df.display()

ReturnDate,TerritoryKey,ProductKey,ReturnQuantity,_rescued_data
1/18/2015,9,312,1,null
1/18/2015,10,310,1,null
1/21/2015,8,346,1,null
1/22/2015,4,311,1,null
2/2/2015,6,312,1,null
2/15/2015,1,312,1,null
2/19/2015,9,311,1,null
2/24/2015,8,314,1,null
3/8/2015,8,350,1,null
3/13/2015,9,350,1,null


### **Drop Rescued Data Column**

In [0]:
df=df.drop("_rescued_data")

### **Checking Schema**

In [0]:
df.printSchema()

root
 |-- ReturnDate: string (nullable = true)
 |-- TerritoryKey: string (nullable = true)
 |-- ProductKey: string (nullable = true)
 |-- ReturnQuantity: string (nullable = true)



### **Fixing ReturnDate column's format**

In [0]:
df=df.withColumn('ReturnDate',regexp_replace(col('ReturnDate'),'/','-'))

In [0]:
df=df.withColumn('ReturnDate',to_date(col("ReturnDate"),'m-d-yyyy'))

### **Fixing TerritoryKey column's format**

In [0]:
df=df.withColumn('TerritoryKey',col('TerritoryKey').cast('int'))

### **Fixing ProductKey column's format**

In [0]:
df=df.withColumn('ProductKey',col('ProductKey').cast('int'))

### **Fixing ReturnQuantity column's format**

In [0]:
df=df.withColumn('ReturnQuantity',col('ReturnQuantity').cast('int'))

### **Get Duplicate rows**

In [0]:
df_duplicates=df.groupBy("ProductKey","ReturnDate","ReturnQuantity","TerritoryKey")\
    .count()\
    .filter(col('count')>1)\
    .drop(col('count'))


df.join(df_duplicates,['ProductKey','ReturnDate','ReturnQuantity','TerritoryKey'],how="inner").display()



ProductKey,ReturnDate,ReturnQuantity,TerritoryKey
530,2016-01-31,1,7
478,2016-01-09,1,1
528,2017-01-27,1,1
215,2017-01-08,1,7
220,2017-01-24,1,4
528,2017-01-09,1,7
535,2016-01-24,1,1
478,2017-01-15,1,10
476,2017-01-08,1,4
477,2017-01-15,1,7


### **Delete Duplicates**

In [0]:
df=df.dropDuplicates(["ProductKey","ReturnDate","ReturnQuantity","TerritoryKey"])

### **Data Quality of KEY columns**

In [0]:
df_territory=spark.sql('''
                       select * from adventure_works.silver.territories
                       ''')

In [0]:
df_product=spark.sql('''
                       select * from adventure_works.silver.products
                       ''')

In [0]:
df=df.filter(
    (col("ProductKey") <= df_product.select(max(col("ProductKey"))).collect()[0][0]) 
    &
    (col("ProductKey") >= df_product.select(min(col("ProductKey"))).collect()[0][0])
    )

In [0]:
df = df.filter(
    (col("TerritoryKey") <= df_territory.select(max(col("SalesTerritoryKey"))).collect()[0][0])
    &    
    (col("TerritoryKey") >= df_territory.select(min(col("SalesTerritoryKey"))).collect()[0][0])
    )

### **Data Writing**

In [0]:
if spark.catalog.tableExists('adventure_works.silver.returns'):

    df_silver_returns =spark.read.table('adventure_works.silver.returns')
    df = df.join(df_silver_returns , ["ProductKey","ReturnDate","ReturnQuantity","TerritoryKey"] , 'left_anti' )

In [0]:
df.write.format('delta').mode('append')\
.save('abfss://silver@adventwork.dfs.core.windows.net/returns')


In [0]:
%sql
create table if not exists adventure_works.silver.returns
using delta
location 'abfss://silver@adventwork.dfs.core.windows.net/returns'

In [0]:
df.display()

ProductKey,ReturnDate,ReturnQuantity,TerritoryKey


In [0]:
%sql
select * from adventure_works.silver.returns

ReturnDate,TerritoryKey,ProductKey,ReturnQuantity
2015-01-13,9,350,1
2015-01-30,4,314,1
2015-01-07,4,342,1
2015-01-10,1,354,1
2015-01-23,9,326,1
2016-01-20,9,377,1
2016-01-21,10,368,1
2016-01-25,7,369,1
2016-01-13,9,371,1
2016-01-22,1,477,1
